# S4 Portfolio — Project Sankey Diagrams

**One page per project.** Each Sankey traces the investment flow through the supply-chain tiers
into sourcing regions / supplying sectors and finally into the three key impact stressors.

| Panel | Stressor | Polarity | Colour |
|-------|----------|----------|--------|
| Left  | GHG tCO₂e | ⬇ negative | red |
| Centre | Employment FTE | ⬆ positive | green |
| Right | Water 000 m³ | ⬇ negative | blue |

**Tier colour key**
- 🟦 Tier 0 — Direct CAPEX (sector breakdown)
- 🟧 Tier 1 — Bilateral first upstream (sourcing-region breakdown)
- 🟩 Tier 2 — Second upstream (sector breakdown)
- 🟪 Tier 3-10 — Deep upstream (sector breakdown)

The width of each band is proportional to the stressor magnitude in that path.

In [1]:
# ══════════════════════════════════════════════════════════════════
# PARAMETERS — edit here
# ══════════════════════════════════════════════════════════════════
from pathlib import Path

RESULTS_DIR  = Path("results")        # supply-chain CSVs
TOP_N        = 5                      # max breakdown nodes per tier
FIGURE_H     = 560                    # figure height in px
EXPORT_HTML  = False                  # set True to save individual HTML files
EXPORT_DIR   = Path("results/sankey_html")

# Stressors shown left→right in each figure
STRESSORS = [
    ("GHG_tCO2e",       "GHG",        "tCO₂e",     "#d62728", "negative"),
    ("Employment_FTE",   "Employment", "FTE",        "#2ca02c", "positive"),
    ("Water_1000m3",     "Water",      "000 m³",     "#1f77b4", "negative"),
]

In [2]:
# ══════════════════════════════════════════════════════════════════
# IMPORTS
# ══════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import display, HTML

# Tier-source colour palette (rgba for link transparency)
TIER_NODE_COL = {
    "t0": "#4e79a7",   # steel blue
    "t1": "#f28e2b",   # amber
    "t2": "#59a14f",   # sage green
    "t3": "#b07aa1",   # muted purple
}
TIER_LINK_COL = {
    "t0": "rgba(78,121,167,0.35)",
    "t1": "rgba(242,142,43,0.35)",
    "t2": "rgba(89,161,79,0.35)",
    "t3": "rgba(176,122,161,0.35)",
}
INTER_COL = {
    "t0": "#aec7e8",
    "t1": "#ffbb78",
    "t2": "#98df8a",
    "t3": "#c5b0d5",
}
print("Imports OK")

Imports OK


In [3]:
# ══════════════════════════════════════════════════════════════════
# LOAD TIER DATA
# ══════════════════════════════════════════════════════════════════
t0 = pd.read_csv(RESULTS_DIR / "supply_chain_tier0.csv")
t1 = pd.read_csv(RESULTS_DIR / "supply_chain_tier1.csv")
t2 = pd.read_csv(RESULTS_DIR / "supply_chain_tier2.csv")
t3 = pd.read_csv(RESULTS_DIR / "supply_chain_tier3_10.csv")

PROJECTS = sorted(t0["project_id"].unique())

# Quick sanity check
for name, df in [("Tier 0", t0), ("Tier 1", t1), ("Tier 2", t2), ("Tier 3-10", t3)]:
    ghg_tot = df.groupby("project_id")["GHG_tCO2e"].sum()
    print(f"{name}: {len(df):>4} rows | GHG range {ghg_tot.min():,.0f}–{ghg_tot.max():,.0f} tCO₂e per project")
print(f"\nProjects ({len(PROJECTS)}): {PROJECTS}")

Tier 0:   72 rows | GHG range 14–376,229 tCO₂e per project
Tier 1:  360 rows | GHG range 9–219,659 tCO₂e per project
Tier 2:  360 rows | GHG range 4–102,160 tCO₂e per project
Tier 3-10:   72 rows | GHG range 4–89,159 tCO₂e per project

Projects (9): ['Hydro_AF', 'Hydro_AS', 'Hydro_EU', 'Proj_001', 'Proj_002', 'Proj_003', 'Rail_EU_DEV', 'Rail_EU_OP1', 'Rail_EU_OP2']


In [4]:
# ══════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════

def _top_n_other(series: pd.Series, n: int = 5) -> pd.Series:
    """Keep top-N by value; bin the rest as 'Other'."""
    series = series[series > 0]
    if len(series) <= n:
        return series
    top = series.nlargest(n)
    rest = series[~series.index.isin(top.index)].sum()
    if rest > 0:
        top = pd.concat([top, pd.Series({"Other": rest})])
    return top


def _build_sankey_data(project_id: str, col: str) -> dict:
    """
    Build node/link arrays for a single stressor Sankey.

    Layer 0 (source): Tier labels
    Layer 1 (middle): T0/T2/T3 → sector breakdown; T1 → sourcing-region breakdown
    Layer 2 (sink):   single terminal node for this stressor
    """
    p0 = t0[t0["project_id"] == project_id]
    p1 = t1[t1["project_id"] == project_id]
    p2 = t2[t2["project_id"] == project_id]
    p3 = t3[t3["project_id"] == project_id]

    nodes   = []          # label list
    n_color = []          # node fill colours
    src     = []
    tgt     = []
    val     = []
    lnk_col = []

    def node(label, color):
        if label not in nodes:
            nodes.append(label)
            n_color.append(color)
        return nodes.index(label)

    def add_links(tier_idx, grp: pd.Series, tier_key: str, prefix: str):
        for name, v in grp.items():
            mid_label = f"{prefix}{name}"
            mid_idx   = node(mid_label, INTER_COL[tier_key])
            col       = TIER_LINK_COL[tier_key]
            # tier → intermediate
            src.append(tier_idx); tgt.append(mid_idx)
            val.append(v);        lnk_col.append(col)
            # intermediate → terminal
            src.append(mid_idx);  tgt.append(term_idx)
            val.append(v);        lnk_col.append(col)

    total = (
        p0[col].sum() + p1[col].sum() +
        p2[col].sum() + p3[col].sum()
    )

    # Tier header nodes
    i0   = node("Tier 0\nDirect CAPEX",         TIER_NODE_COL["t0"])
    i1   = node("Tier 1\nBilateral Sourcing",   TIER_NODE_COL["t1"])
    i2   = node("Tier 2\nSecond Upstream",      TIER_NODE_COL["t2"])
    i3   = node("Tier 3–10\nDeep Upstream",     TIER_NODE_COL["t3"])

    # Terminal node (placed after tier nodes so index is stable)
    _, sname, sunit, s_col, _ = next(s for s in STRESSORS if s[0] == col)
    term_label = f"{sname}\n{total:,.0f} {sunit}"
    term_idx   = node(term_label, s_col)

    # Tier 0: sector breakdown
    grp0 = _top_n_other(p0.groupby("supplying_sector")[col].sum(), TOP_N)
    add_links(i0, grp0, "t0", "T0 ")

    # Tier 1: sourcing-region breakdown (bilateral)
    grp1 = _top_n_other(p1.groupby("sourcing_region")[col].sum(), TOP_N)
    add_links(i1, grp1, "t1", "T1 ")

    # Tier 2: sector breakdown
    grp2 = _top_n_other(p2.groupby("supplying_sector")[col].sum(), TOP_N)
    add_links(i2, grp2, "t2", "T2 ")

    # Tier 3-10: sector breakdown
    grp3 = _top_n_other(p3.groupby("supplying_sector")[col].sum(), TOP_N)
    add_links(i3, grp3, "t3", "T3+ ")

    return dict(
        nodes=nodes, n_color=n_color,
        src=src, tgt=tgt, val=val, lnk_col=lnk_col,
        total=total, term_label=term_label,
    )


def build_project_figure(project_id: str) -> go.Figure:
    """
    Assemble a three-panel Sankey figure (GHG | Employment | Water)
    for a single project.
    """
    n_panels = len(STRESSORS)
    gap      = 0.03
    width    = (1.0 - gap * (n_panels - 1)) / n_panels

    fig = go.Figure()

    for i, (col, sname, sunit, s_col, polarity) in enumerate(STRESSORS):
        d    = _build_sankey_data(project_id, col)
        x0   = i * (width + gap)
        x1   = x0 + width
        sign = "[−]" if polarity == "negative" else "[+]"

        fig.add_trace(go.Sankey(
            domain=dict(x=[x0, x1], y=[0.0, 0.92]),
            arrangement="snap",
            node=dict(
                label=d["nodes"],
                color=d["n_color"],
                pad=14,
                thickness=18,
                line=dict(color="white", width=0.4),
            ),
            link=dict(
                source=d["src"],
                target=d["tgt"],
                value=d["val"],
                color=d["lnk_col"],
            ),
        ))

        # Panel title as annotation
        fig.add_annotation(
            x=(x0 + x1) / 2, y=1.0,
            xref="paper", yref="paper",
            text=f"<b>{sign} {sname}</b><br><span style='font-size:11px;color:{s_col}'>"
                 f"{d['total']:,.0f} {sunit}</span>",
            showarrow=False,
            font=dict(size=13),
            align="center",
        )

    # Project-level title
    asset  = t0.loc[t0["project_id"] == project_id, "asset_class"].iloc[0]
    region = t0.loc[t0["project_id"] == project_id, "region"].iloc[0]

    fig.update_layout(
        title=dict(
            text=(
                f"<b>{project_id}</b>  ·  {asset}  ·  {region}"
                "<br><span style='font-size:11px;color:#555'>"
                "Investment → Tier 0–10 supply-chain → impact stressors"
                "</span>"
            ),
            x=0.01, xanchor="left",
            font=dict(size=15),
        ),
        height=FIGURE_H,
        margin=dict(l=20, r=20, t=90, b=20),
        paper_bgcolor="#fafafa",
        font=dict(family="Arial", size=10),
    )
    return fig

print("Helper functions defined.")

Helper functions defined.


In [5]:
# ══════════════════════════════════════════════════════════════════
# TIER + REGION LEGEND  (printed once, applies to all figures)
# ══════════════════════════════════════════════════════════════════
legend_html = """
<div style='font-family:Arial; font-size:12px; margin:8px 0;'>
<b>Tier colour key:</b>&nbsp;
<span style='background:#4e79a7;color:white;padding:2px 6px;border-radius:3px'>Tier 0 Direct CAPEX</span>&nbsp;
<span style='background:#f28e2b;color:white;padding:2px 6px;border-radius:3px'>Tier 1 Bilateral Sourcing</span>&nbsp;
<span style='background:#59a14f;color:white;padding:2px 6px;border-radius:3px'>Tier 2 Second Upstream</span>&nbsp;
<span style='background:#b07aa1;color:white;padding:2px 6px;border-radius:3px'>Tier 3–10 Deep Upstream</span>
&emsp;<b>Stressor panels:</b>&nbsp;
<span style='color:#d62728'><b>[−] GHG</b></span>&nbsp;
<span style='color:#2ca02c'><b>[+] Jobs</b></span>&nbsp;
<span style='color:#1f77b4'><b>[−] Water</b></span>
</div>
"""
display(HTML(legend_html))

In [6]:
# ══════════════════════════════════════════════════════════════════
# MAIN LOOP — one figure per project
# ══════════════════════════════════════════════════════════════════
if EXPORT_HTML:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for pid in PROJECTS:
    fig = build_project_figure(pid)
    fig.show()

    if EXPORT_HTML:
        out_path = EXPORT_DIR / f"{pid}_sankey.html"
        fig.write_html(str(out_path), include_plotlyjs="cdn")
        print(f"  Saved → {out_path}")

In [7]:
# ══════════════════════════════════════════════════════════════════
# PORTFOLIO SUMMARY — GHG by tier × project (stacked bar + deep-upstream callout)
# ══════════════════════════════════════════════════════════════════
import plotly.graph_objects as go

tier_sums = {}
for pid in PROJECTS:
    tier_sums[pid] = {
        "Tier 0":     t0[t0["project_id"] == pid]["GHG_tCO2e"].sum(),
        "Tier 1":     t1[t1["project_id"] == pid]["GHG_tCO2e"].sum(),
        "Tier 2":     t2[t2["project_id"] == pid]["GHG_tCO2e"].sum(),
        "Tier 3-10":  t3[t3["project_id"] == pid]["GHG_tCO2e"].sum(),
    }

tier_df = pd.DataFrame(tier_sums).T
tier_df["Total"] = tier_df.sum(axis=1)
tier_df = tier_df.sort_values("Total", ascending=True)

bar_cols = ["#4e79a7", "#f28e2b", "#59a14f", "#b07aa1"]

fig2 = go.Figure()
for col, color in zip(["Tier 0", "Tier 1", "Tier 2", "Tier 3-10"], bar_cols):
    fig2.add_trace(go.Bar(
        name=col,
        y=tier_df.index,
        x=tier_df[col],
        orientation="h",
        marker_color=color,
        text=tier_df[col].apply(lambda v: f"{v:,.0f}" if v > 500 else ""),
        textposition="inside",
        insidetextanchor="middle",
        textfont=dict(color="white", size=9),
    ))

# Deep-upstream share annotation
for pid in tier_df.index:
    total = tier_df.loc[pid, "Total"]
    deep  = tier_df.loc[pid, "Tier 3-10"]
    if total > 0:
        fig2.add_annotation(
            x=total * 1.01, y=pid,
            text=f"Deep: {deep/total*100:.0f}%",
            showarrow=False,
            font=dict(size=9, color="#b07aa1"),
            xanchor="left",
        )

fig2.update_layout(
    title="<b>Portfolio GHG by Tier</b> — purple = Deep Upstream (Tier 3-10) share",
    barmode="stack",
    xaxis_title="GHG tCO₂e",
    height=380,
    legend=dict(orientation="h", y=1.05),
    margin=dict(l=120, r=120, t=60, b=40),
    paper_bgcolor="#fafafa",
    font=dict(family="Arial", size=11),
)
fig2.show()

In [8]:
# ══════════════════════════════════════════════════════════════════
# TIER 1 BILATERAL SOURCING — GHG heatmap: project × sourcing region
# Shows exactly where the first-upstream carbon physically comes from
# ══════════════════════════════════════════════════════════════════
import plotly.figure_factory as ff

piv = (
    t1.groupby(["project_id", "sourcing_region"])["GHG_tCO2e"]
    .sum()
    .unstack(fill_value=0)
)
# Sort projects by total GHG descending
piv = piv.loc[piv.sum(axis=1).sort_values(ascending=False).index]

fig3 = go.Figure(go.Heatmap(
    z=piv.values,
    x=piv.columns.tolist(),
    y=piv.index.tolist(),
    colorscale="YlOrRd",
    text=[[f"{v:,.0f}" for v in row] for row in piv.values],
    texttemplate="%{text}",
    textfont=dict(size=10),
    colorbar=dict(title="GHG tCO₂e"),
))
fig3.update_layout(
    title="<b>Tier 1 GHG Sourcing Heatmap</b> — project × sourcing region (bilateral)",
    xaxis_title="Sourcing Region",
    yaxis_title="Project",
    height=340,
    margin=dict(l=130, r=40, t=60, b=60),
    paper_bgcolor="#fafafa",
    font=dict(family="Arial", size=11),
)
fig3.show()

In [9]:
# ══════════════════════════════════════════════════════════════════
# TRADE-OFF SCATTER — net GHG (all tiers) vs Total Jobs
# Bubble size = Water footprint; quadrant lines at portfolio medians
# ══════════════════════════════════════════════════════════════════
rows = []
for pid in PROJECTS:
    ghg = (t0[t0["project_id"]==pid]["GHG_tCO2e"].sum() +
           t1[t1["project_id"]==pid]["GHG_tCO2e"].sum() +
           t2[t2["project_id"]==pid]["GHG_tCO2e"].sum() +
           t3[t3["project_id"]==pid]["GHG_tCO2e"].sum())
    emp = (t0[t0["project_id"]==pid]["Employment_FTE"].sum() +
           t1[t1["project_id"]==pid]["Employment_FTE"].sum() +
           t2[t2["project_id"]==pid]["Employment_FTE"].sum() +
           t3[t3["project_id"]==pid]["Employment_FTE"].sum())
    wat = (t0[t0["project_id"]==pid]["Water_1000m3"].sum() +
           t1[t1["project_id"]==pid]["Water_1000m3"].sum() +
           t2[t2["project_id"]==pid]["Water_1000m3"].sum() +
           t3[t3["project_id"]==pid]["Water_1000m3"].sum())
    asset  = t0.loc[t0["project_id"]==pid, "asset_class"].iloc[0]
    region = t0.loc[t0["project_id"]==pid, "region"].iloc[0]
    rows.append(dict(project_id=pid, GHG=ghg, Employment=emp, Water=wat,
                     asset_class=asset, region=region))

sc_df = pd.DataFrame(rows)
med_ghg = sc_df["GHG"].median()
med_emp = sc_df["Employment"].median()

asset_colors = {"Energy": "#d62728", "Health": "#2ca02c", "Transport": "#1f77b4"}
sc_df["color"] = sc_df["asset_class"].map(asset_colors).fillna("#7f7f7f")
sc_df["bubble"] = (sc_df["Water"] / sc_df["Water"].max() * 60 + 8).clip(lower=8)

fig4 = go.Figure()
for asset, grp in sc_df.groupby("asset_class"):
    fig4.add_trace(go.Scatter(
        x=grp["GHG"], y=grp["Employment"],
        mode="markers+text",
        name=asset,
        marker=dict(
            size=grp["bubble"],
            color=asset_colors.get(asset, "#7f7f7f"),
            opacity=0.75,
            line=dict(color="white", width=1),
        ),
        text=grp["project_id"],
        textposition="top center",
        textfont=dict(size=9),
    ))

# Quadrant lines
for val, axis, dash, color, label in [
    (med_ghg, "x", "dot", "#d62728", "median GHG"),
    (med_emp, "y", "dot", "#2ca02c", "median Jobs"),
]:
    kw = dict(x0=val, x1=val, y0=0, y1=1, yref="paper") if axis=="x" else \
         dict(y0=val, y1=val, x0=0, x1=1, xref="paper")
    fig4.add_shape(type="line", line=dict(dash=dash, color=color, width=1.5), **kw)

fig4.add_annotation(x=med_ghg*0.05, y=sc_df["Employment"].max()*0.95,
    text="← Low GHG, High Jobs", showarrow=False,
    font=dict(size=10, color="#2ca02c"), xanchor="left")
fig4.add_annotation(x=sc_df["GHG"].max()*0.85, y=med_emp*0.1,
    text="High GHG, Low Jobs →", showarrow=False,
    font=dict(size=10, color="#d62728"), xanchor="right")

fig4.update_layout(
    title="<b>Trade-off Quadrant</b> — net GHG vs total Employment (bubble = Water footprint)",
    xaxis_title="Supply-chain GHG tCO₂e [−]",
    yaxis_title="Supply-chain Employment FTE [+]",
    height=480,
    margin=dict(l=80, r=40, t=70, b=60),
    paper_bgcolor="#fafafa",
    font=dict(family="Arial", size=11),
    legend=dict(title="Asset class"),
)
fig4.show()